In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df=pd.read_excel(r"C:\Users\DEEPA\Downloads\NGO Teacher training analysis\final ngo cleaned data.xlsx",
                 sheet_name='verified_feedback_fact')
df.head()

,Date,Name,School,Designation,Workshop_org_feedback,Relevance_of_topics_to_interest,Science-Communication_session_Rating,Science-Communication_session_Feedback,AI_Tools_Session_Rating,AI_Tools_Session_Feedback,...,TimeMgmt_&_Flow_of_Sess_Rating,Robotics_Session_Ratings.1,Robotics_Session_Feedback.1,Climate_Change_Rating,Session_Engagement_Ratings,Faciliatator_Knowledge_Ratings,TopicSuggestion_for_future_workshops,Improvement_Suggestions,Future Topic Theme,Improvement Theme
0,2025-10-16,Ghulam Nabi Bhat,Bms Gunbagh,Teacher,Good,Good,Good,NaN,Excellent,NaN,...,Good,Excellent,NaN,Excellent,Agree,Agree,Environmental,Time was short ( better 2 or 3 days),Climate & Environment,Other
1,2025-10-16,Riyaz Ahmed Bhat,Gms Khojabagh,Teacher,Excellent,NaN,Excellent,NaN,NaN,NaN,...,Excellent,Excellent,NaN,NaN,Strongly Agree,Strongly Agree,"AI Tools, lab visits, Refresher Courses for Te...",NaN,AI & Technology,No Suggestion
2,2025-10-16,Imtiyaz Hussain,Ms Motii Mohalla,Teacher,Good,Good,Excellent,NaN,Good,NaN,...,Good,Good,NaN,Excellent,Agree,Agree,There must be practical session for AI based A...,NaN,AI & Technology,No Suggestion
3,2025-10-16,Mehraj Uddin,Gms Brane,Teacher,Good,Fair,Excellent,NaN,Fair,NaN,...,Good,Good,NaN,Excellent,Agree,Agree,NaN,NaN,Other,No Suggestion
4,2025-10-16,Rehana Sediq,Gms Amlazi Zone Hamal,General Line Teacher,Good,Fair,Excellent,NaN,Good,NaN,...,Excellent,Good,NaN,NaN,Strongly Agree,Strongly Agree,"More expanded workshop on AI, ChatGPT should b...","There is always a chance of improvement, keep ...",AI & Technology,Other


In [3]:
df.columns

Index(['Date', 'Name', 'School', 'Designation', 'Workshop_org_feedback',
       'Relevance_of_topics_to_interest',
       'Science-Communication_session_Rating',
       'Science-Communication_session_Feedback', 'AI_Tools_Session_Rating',
       'AI_Tools_Session_Feedback', 'AI_Automation_session_rating',
       'AI_automation_Feedback', 'Climate_Change_session_Feedback',
       'Enhanced_Understanding_Ratings', 'Gained_new_tools',
       'TimeMgmt_&_Flow_of_Sess_Rating', 'Robotics_Session_Ratings.1',
       'Robotics_Session_Feedback.1', 'Climate_Change_Rating',
       'Session_Engagement_Ratings', 'Faciliatator_Knowledge_Ratings',
       'TopicSuggestion_for_future_workshops', 'Improvement_Suggestions',
       'Future Topic Theme', 'Improvement Theme'],
      dtype='object')

In [4]:
df.isna().sum()

Date                                       0
Name                                       0
School                                     0
Designation                                2
Workshop_org_feedback                      1
Relevance_of_topics_to_interest            4
Science-Communication_session_Rating       0
Science-Communication_session_Feedback    62
AI_Tools_Session_Rating                    8
AI_Tools_Session_Feedback                 67
AI_Automation_session_rating               2
AI_automation_Feedback                    68
Climate_Change_session_Feedback           77
Enhanced_Understanding_Ratings             3
Gained_new_tools                           4
TimeMgmt_&_Flow_of_Sess_Rating             2
Robotics_Session_Ratings.1                 3
Robotics_Session_Feedback.1               72
Climate_Change_Rating                     24
Session_Engagement_Ratings                 6
Faciliatator_Knowledge_Ratings             3
TopicSuggestion_for_future_workshops      25
Improvemen

In [5]:
#Create combined session feedback
df['combined_feedback'] = (
    df['Science-Communication_session_Feedback'].fillna('') + ' ' +
    
    df['AI_Tools_Session_Feedback'].fillna('') + ' ' +
    
    df['AI_automation_Feedback'].fillna('') + ' ' +
    
    df['Climate_Change_session_Feedback'].fillna('') + ' ' +
    
    df['Robotics_Session_Feedback.1'].fillna('') + ' ' +
    
    df['TopicSuggestion_for_future_workshops'].fillna('') + ' ' +
    
    df['Improvement_Suggestions'].fillna('')
)

In [6]:
#clean text
def clean_text(text):
    text = str(text).lower()
    
    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['clean_feedback'] = df['combined_feedback'].apply(clean_text)

In [7]:
df['clean_feedback'].head()

0          environmental time was short better or days
1    ai tools lab visits refresher courses for teac...
2    there must be practical session for ai based a...
3                                                     
4    more expanded workshop on ai chatgpt should be...
Name: clean_feedback, dtype: object

In [8]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

In [9]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\DEEPA\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [10]:
#intialise sentiment anlyser
sia = SentimentIntensityAnalyzer()

In [11]:
#generate sentiment score
df['sentiment_score'] = df['clean_feedback'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

In [12]:
df['sentiment_score'].head()

0    0.4404
1    0.0000
2    0.0000
3    0.0000
4    0.6124
Name: sentiment_score, dtype: float64

In [13]:
def sentiment_label(score):
    if score>=0.5 :
        return 'Positive'
    elif score<=-0.5:
        return 'Negative'
    else:
        return 'Neutral'
df['sentiment_category']=df['sentiment_score'].apply(sentiment_label)

In [14]:
df['sentiment_category'].head()

0     Neutral
1     Neutral
2     Neutral
3     Neutral
4    Positive
Name: sentiment_category, dtype: object

In [15]:
print(df[
    [
        'combined_feedback',
        'sentiment_score',
        'sentiment_category'
    ]
].head())

                                   combined_feedback  sentiment_score  \
0       Environmental Time was short ( better 2 o...           0.4404   
1       AI Tools, lab visits, Refresher Courses f...           0.0000   
2       There must be practical session for AI ba...           0.0000   
3                                                              0.0000   
4       More expanded workshop on AI, ChatGPT sho...           0.6124   

  sentiment_category  
0            Neutral  
1            Neutral  
2            Neutral  
3            Neutral  
4           Positive  


In [26]:
df['Science_Sentiment'] = df['Science-Communication_session_Feedback'].fillna('').apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

df['AI_Tools_Sentiment'] = df['AI_Tools_Session_Feedback'].fillna('').apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

df['AI_Automation_Sentiment'] = df['AI_automation_Feedback'].fillna('').apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

df['Climate_Change_Sentiment'] = df['Climate_Change_session_Feedback'].fillna('').apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

df['Robotics_Sentiment'] = df['Robotics_Session_Feedback.1'].fillna('').apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

In [27]:
df['Science_Sentiment'] .head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Science_Sentiment, dtype: float64

In [28]:
df.to_excel(
    "ngo_sentiment_output.xlsx",
    index=False
)